# LLM World tracking problem exploration

In [ ]:
import pandas as pd

from reflect.core.paths import sim_results_root

LABELING_RESULTS_PATH = str(sim_results_root() / "error_taxonomy_full.csv")
EPISODE_RUN_DATA_PATH = str(sim_results_root() / "reflect_results_run4.json")

In [ ]:
# Load and filter episodes with reasoning failures
labeled_episodes_df = pd.read_csv(LABELING_RESULTS_PATH)
reasoning_failed_episodes = labeled_episodes_df[labeled_episodes_df["taxonomy_code"].isin(["RSN-DIAG", "RSN-PLAN"])]

In [ ]:
# Load and filter episode data to match the reasoning failed episodes
with open(EPISODE_RUN_DATA_PATH, 'r') as f:
    episode_run_data = pd.read_json(f).drop(columns=['status', 'num_keyframes', 'num_events', 'global_sg', 'error'], errors='ignore')

episode_run_data.head()

In [ ]:
# Keep only the reasoning-failure episodes and unfold the nested prompt log
reasoning_episode_run_data = reasoning_failed_episodes.merge(
    episode_run_data,
    on=["episode", "task"],
    how="left",
    validate="one_to_one",
).copy()

reasoning_episode_run_data["episode_num"] = (
    reasoning_episode_run_data["episode"]
    .str.extract(r"-(\d+)$")
    .astype(int)
)
reasoning_episode_run_data["prompt_count"] = reasoning_episode_run_data["prompts_log"].apply(
    lambda prompts: len(prompts) if isinstance(prompts, list) else 0
)
reasoning_episode_run_data["prompt_calls"] = reasoning_episode_run_data["prompts_log"].apply(
    lambda prompts: [entry.get("call") for entry in prompts] if isinstance(prompts, list) else []
)
reasoning_episode_run_data = reasoning_episode_run_data.sort_values(["task", "episode_num"]).reset_index(drop=True)

prompt_rows = []
for row in reasoning_episode_run_data.to_dict("records"):
    for prompt_idx, prompt_entry in enumerate(row.get("prompts_log") or [], start=1):
        prompt = prompt_entry.get("prompt") or {}
        prompt_rows.append(
            {
                "episode": row["episode"],
                "task": row["task"],
                "taxonomy_code": row["taxonomy_code"],
                "category": row["category"],
                "prompt_idx": prompt_idx,
                "call": prompt_entry.get("call"),
                "system_prompt": prompt.get("system"),
                "user_prompt": prompt.get("user"),
                "response": prompt_entry.get("response"),
            }
        )

reasoning_episode_prompts = pd.DataFrame(prompt_rows)

reasoning_episode_run_data[["episode", "taxonomy_code", "category", "prompt_count", "prompt_calls"]]


In [ ]:
from IPython.display import Markdown, display


def clip_text(text, max_chars=None):
    text = "" if text is None else str(text).strip()
    if max_chars is not None and len(text) > max_chars:
        return text[:max_chars].rstrip() + "\n\n[truncated]"
    return text


def format_action_step(step):
    if isinstance(step, dict):
        action = step.get("action", "?")
        args = [step.get("obj1"), step.get("obj2")]
        args = [str(arg) for arg in args if arg not in (None, "")]
        return f"({action}{', ' if args else ''}{', '.join(args)})"
    return str(step)


def format_plan(plan):
    plan = plan or []
    if not plan:
        return "_None_"
    return "\n".join(f"{idx}. {format_action_step(step)}" for idx, step in enumerate(plan, start=1))


def fenced_block(text):
    text = "" if text is None else str(text)
    return f"```text\n{text.rstrip()}\n```"


def render_episode_markdown(row, include_l1=False, include_l2=True, max_prompt_chars=None):
    reasoning = row.get("reasoning_dict") or {}
    replan = row.get("replan_dict") or {}
    correction = row.get("correction_dict") or {}
    prompts = row.get("prompts_log") or []

    parts = [
        f"## {row['episode']}",
        f"**Task:** `{row['task']}`  ",
        f"**Taxonomy:** `{row['taxonomy_code']}` ({row['category']})  ",
        f"**Annotated success:** `{row['c_success']}`",
        "",
        "### Failure summary",
        f"- Predicted error type: `{reasoning.get('error_type', 'N/A')}`",
        f"- Ground-truth error type: `{reasoning.get('gt_error_type', 'N/A')}`",
        f"- Predicted failure step(s): `{reasoning.get('pred_failure_step', 'N/A')}`",
        f"- Ground-truth failure step: `{reasoning.get('gt_failure_step', 'N/A')}`",
        "",
        "**Predicted failure reason**",
        fenced_block(reasoning.get("pred_failure_reason", "")),
        "",
        "**Ground-truth failure reason**",
        fenced_block(reasoning.get("gt_failure_reason", "")),
        "",
        "### Plans",
        "**Task plan**",
        fenced_block(format_plan(replan.get("task_plan"))),
        "",
        "**LLM replan**",
        fenced_block(format_plan(replan.get("llm_plan"))),
        "",
        f"**Correction success:** `{correction.get('success', 'N/A')}`",
        "**Correction plan**",
        fenced_block(format_plan(correction.get("llm_plan"))),
    ]

    if include_l2:
        parts.extend(
            [
                "",
                "<details><summary>L2 summary</summary>",
                "",
                fenced_block(clip_text(row.get("l2_summary", ""), max_prompt_chars)),
                "",
                "</details>",
            ]
        )

    if include_l1:
        parts.extend(
            [
                "",
                "<details><summary>L1 summary</summary>",
                "",
                fenced_block(clip_text(row.get("l1_summary", ""), max_prompt_chars)),
                "",
                "</details>",
            ]
        )

    parts.extend(["", "### Prompt log"])
    for prompt_idx, prompt_entry in enumerate(prompts, start=1):
        prompt = prompt_entry.get("prompt") or {}
        call = prompt_entry.get("call", f"prompt-{prompt_idx}")
        parts.extend(
            [
                "",
                f"<details><summary>Prompt {prompt_idx}: {call}</summary>",
                "",
                "**System prompt**",
                fenced_block(clip_text(prompt.get("system", ""))),
                "",
                "**User prompt**",
                fenced_block(clip_text(prompt.get("user", ""), max_prompt_chars)),
                "",
                "**Response**",
                fenced_block(clip_text(prompt_entry.get("response", ""), max_prompt_chars)),
                "",
                "</details>",
            ]
        )

    return "\n".join(parts)


def show_reasoning_episode(episode_id, include_l1=False, include_l2=True, max_prompt_chars=None):
    matches = reasoning_episode_run_data.loc[reasoning_episode_run_data["episode"] == episode_id]
    if matches.empty:
        raise KeyError(f"Episode not found: {episode_id}")
    row = matches.iloc[0].to_dict()
    display(Markdown(render_episode_markdown(row, include_l1=include_l1, include_l2=include_l2, max_prompt_chars=max_prompt_chars)))


def show_all_reasoning_episodes(include_l1=False, include_l2=True, max_prompt_chars=None):
    for row in reasoning_episode_run_data.to_dict("records"):
        display(Markdown(render_episode_markdown(row, include_l1=include_l1, include_l2=include_l2, max_prompt_chars=max_prompt_chars)))


In [ ]:
# Example usage
show_reasoning_episode("makeCoffee-6", max_prompt_chars=None)


## Reasoning Effort Sweep

Per the OpenAI reasoning guide and the current GPT-5.4 model docs, GPT-5.4 supports reasoning effort modes `none`, `low`, `medium`, `high`, and `xhigh`.

This section re-runs just the RSN-labeled episodes across those modes and compares Loc / Co-plan automatically, while preparing a combined Exp annotation table for manual scoring.


In [ ]:
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from main.reasoning_effort_eval import (
    build_baseline_rsn_summary,
    build_exp_annotation_table,
    metric_long_format,
    results_to_metrics_frame,
    run_reasoning_effort_sweep,
    summarize_metric_scaling,
    sweep_output_dir,
)

OPENAI_MODEL = "gpt-5.4"
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]  # set in your shell or .env (see .env.example)
WITH_AUDIO = 1
SIM_DATA_DIR = "./datasets/sim_data"
RESULTS_DIR = str(sim_results_root())
MAX_WORKERS = 2
REASONING_EFFORTS = ["none", "low", "medium", "high", "xhigh"]
REASONING_SWEEP_DIR = sweep_output_dir(RESULTS_DIR)
REASONING_EXP_EVAL_PATH = os.path.join(
    REASONING_SWEEP_DIR,
    f"exp_eval__{OPENAI_MODEL.replace('.', '_')}.csv",
)

if 'prompt_template' not in locals():
    with open(os.path.join("./LLM", "prompts.json")) as _f:
        prompt_template = json.load(_f)

reasoning_eval_episodes = reasoning_failed_episodes[
    ["task", "episode", "taxonomy_code", "category", "c_success"]
].drop_duplicates().copy()

print(f"RSN episodes queued: {len(reasoning_eval_episodes)}")
print("Reasoning efforts:", REASONING_EFFORTS)
reasoning_eval_episodes[["episode", "taxonomy_code", "category"]]


In [ ]:
# Expensive cell: this makes OpenAI API calls and caches each effort run under
# results/sim_data/reasoning_effort_sweep/.
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is required before running the sweep.")
if len(REASONING_EFFORTS) != 5:
    raise ValueError(f"Expected 5 reasoning efforts for GPT-5.4, got: {REASONING_EFFORTS}")
print("Running efforts:", REASONING_EFFORTS)

reasoning_effort_results = run_reasoning_effort_sweep(
    reasoning_eval_episodes,
    api_key=OPENAI_API_KEY,
    model=OPENAI_MODEL,
    prompt_template=prompt_template,
    sim_data_dir=SIM_DATA_DIR,
    results_dir=RESULTS_DIR,
    reasoning_efforts=REASONING_EFFORTS,
    with_audio=WITH_AUDIO,
    max_workers=MAX_WORKERS,
    load_existing=True,
    save_results=True,
    rerun_errors=True,
)

list(reasoning_effort_results.keys())


In [ ]:
reasoning_effort_metrics = results_to_metrics_frame(reasoning_effort_results).merge(
    reasoning_eval_episodes[["task", "episode", "taxonomy_code", "category"]],
    on=["task", "episode"],
    how="left",
)

baseline_rsn_summary = build_baseline_rsn_summary(
    error_taxonomy_all_path=str(sim_results_root() / "error_taxonomy_all_run4.csv"),
    error_taxonomy_full_path=str(sim_results_root() / "error_taxonomy_full.csv"),
)

exp_eval_table = build_exp_annotation_table(
    reasoning_effort_metrics,
    output_path=REASONING_EXP_EVAL_PATH,
)
exp_scores_df = pd.read_csv(REASONING_EXP_EVAL_PATH) if os.path.exists(REASONING_EXP_EVAL_PATH) else pd.DataFrame()

reasoning_effort_summary = summarize_metric_scaling(
    reasoning_effort_metrics,
    exp_scores_df=exp_scores_df,
    baseline_summary_df=baseline_rsn_summary,
)

print(f"Exp annotation file: {REASONING_EXP_EVAL_PATH}")
print(f"Scored Exp rows: {exp_scores_df['human_score'].count() if 'human_score' in exp_scores_df.columns else 0}")
reasoning_effort_summary


In [ ]:
metric_cols = [
    ("loc_rate", "Loc", "steelblue"),
    ("coplan_rate", "Co-plan", "seagreen"),
    ("exp_rate", "Exp", "darkorange"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"RSN Episodes: metric scaling by reasoning effort ({OPENAI_MODEL})", fontsize=16, fontweight="bold")

for ax, (column, label, color) in zip(axes, metric_cols):
    plot_df = reasoning_effort_summary[["effort", column]].dropna(subset=[column]).copy()
    baseline_df = plot_df[plot_df["effort"] == "baseline_run4"]
    effort_df = plot_df[plot_df["effort"] != "baseline_run4"].reset_index(drop=True)
    x = np.arange(len(effort_df))

    ax.plot(x, effort_df[column], color=color, marker="o", linewidth=2.5, markersize=7, label="Reasoning effort")
    ax.scatter(x, effort_df[column], color=color, s=55, zorder=3)

    if not baseline_df.empty:
        baseline_value = baseline_df[column].iloc[0]
        ax.axhline(baseline_value, color="#444444", linestyle="--", linewidth=1.5, alpha=0.9, label="Baseline run4")
        ax.text(
            x[0] -0.15 if len(x) else 0,
            baseline_value,
            f"baseline {baseline_value:.1f}%",
            ha="left",
            va="bottom",
            fontsize=9,
            color="#444444",
        )

    ax.set_title(label, fontsize=15)
    ax.set_ylim(0, 105)
    if column == "loc_rate":
        ax.set_ylabel("Success rate (%)", fontsize=15)
    ax.set_xticks(x)
    ax.set_xticklabels(effort_df["effort"], rotation=30, fontsize=12)
    ax.grid(axis="y", linestyle=":", alpha=0.3)

    for x_i, value in zip(x, effort_df[column]):
        ax.text(x_i, value + 1, f"{value:.1f}%", ha="center", va="bottom", fontsize=12)

    if column == "coplan_rate":
        ax.legend(frameon=False, fontsize=12, loc="upper right")
    else:
        ax.legend(frameon=False, fontsize=12, loc="lower right")

plt.tight_layout()
plt.show()

reasoning_effort_by_type = (
    reasoning_effort_metrics.groupby(["effort", "gt_error_type"], as_index=False)
    .agg(
        episodes=("episode", "count"),
        loc_rate=("loc_hit", lambda s: s.mean() * 100),
        coplan_rate=("coplan_success", lambda s: s.mean() * 100),
        status_ok_rate=("status_ok", lambda s: s.mean() * 100),
    )
    .sort_values(["gt_error_type", "effort"])
    .reset_index(drop=True)
)
